# 🏠 House Price Prediction — V14 (Triple Ensemble + Stacking + Deep Text Features + GPU)
**Các cải tiến so với V11**:
- **CatBoost** thay thế LightGBM trong ensemble hoặc thêm vào triple ensemble
- **Stacking** với meta-model thay vì blending thủ công
- **TF-IDF + SVD** trích xuất 50 chiều latent từ mô tả
- **Ward-level statistics** (PPM median, price volatility)
- **GroupKFold CV** theo location cluster để tránh data leakage
- **Isolation Forest** cho outlier detection tinh vi
- **Optuna với nhiều tham số hơn** (gamma, reg_alpha, reg_lambda, lambda_l1, lambda_l2)
- **SHAP feature selection** tự động loại bỏ features không quan trọng
- **GPU Acceleration** nếu có CUDA (XGBoost, LightGBM, CatBoost)

## 1. Install packages

In [13]:
!pip install xgboost lightgbm catboost category_encoders underthesea optuna google-cloud-bigquery scikit-learn shap -qq

## 2. Imports

In [14]:
import pandas as pd
import numpy as np
import joblib
import json
import optuna
import re
import warnings
import xgboost as xgb
import lightgbm as lgb
import category_encoders as ce
from catboost import CatBoostRegressor
from google.cloud import bigquery
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import IsolationForest
from underthesea import text_normalize

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── GPU Detection ────────────────────────────────────────────────────────
def detect_gpu():
    """Detect available GPU and return appropriate device strings for each model.
    Verifies that each library actually supports GPU before enabling it.
    """
    gpu_available = False
    try:
        import torch
        if torch.cuda.is_available():
            gpu_available = True
            print(f'✅ GPU detected: {torch.cuda.get_device_name(0)}')
    except ImportError:
        pass

    if not gpu_available:
        print('ℹ GPU not available, using CPU mode')
        return {
            'xgb_tree_method': 'hist',
            'xgb_device': 'cpu',
            'lgb_device': 'cpu',
            'cat_task_type': 'CPU'
        }

    # ── Verify XGBoost GPU support ────────────────────────────────────
    xgb_device = 'cpu'
    try:
        import xgboost as xgb
        # XGBoost >= 2.0: use device='cuda' instead of tree_method='gpu_hist'
        import numpy as np
        X_dummy = np.random.rand(10, 5)
        y_dummy = np.random.rand(10)
        model = xgb.XGBRegressor(device='cuda', tree_method='hist', n_estimators=1, verbosity=0)
        model.fit(X_dummy, y_dummy)
        xgb_device = 'cuda'
        print('  ✅ XGBoost GPU support verified (device=cuda)')
    except Exception as e:
        print(f'  ⚠ XGBoost GPU not supported ({str(e)[:60]}...), using cpu')

    # ── Verify LightGBM GPU support ───────────────────────────────────────
    lgb_device = 'cpu'
    try:
        import lightgbm as lgb
        import numpy as np
        X_dummy = np.random.rand(10, 5)
        y_dummy = np.random.rand(10)
        model = lgb.LGBMRegressor(device='gpu', n_estimators=1, verbose=-1)
        model.fit(X_dummy, y_dummy)
        lgb_device = 'gpu'
        print('  ✅ LightGBM GPU support verified')
    except Exception as e:
        print(f'  ⚠ LightGBM GPU not supported ({str(e)[:60]}...), using cpu')

    # ── Verify CatBoost GPU support ───────────────────────────────────────
    cat_task_type = 'CPU'
    try:
        from catboost import CatBoostRegressor
        import numpy as np
        X_dummy = np.random.rand(10, 5)
        y_dummy = np.random.rand(10)
        model = CatBoostRegressor(task_type='GPU', iterations=1, verbose=0)
        model.fit(X_dummy, y_dummy)
        cat_task_type = 'GPU'
        print('  ✅ CatBoost GPU support verified')
    except Exception as e:
        print(f'  ⚠ CatBoost GPU not supported ({str(e)[:60]}...), using CPU')

    return {
        'xgb_tree_method': 'hist',
        'xgb_device': xgb_device,
        'lgb_device': lgb_device,
        'cat_task_type': cat_task_type
    }

GPU_CONFIG = detect_gpu()
print('✅ All packages imported successfully')

## 3. Load data from BigQuery

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
GCP_KEY_PATH  = '/lakehouse/default/Files/real-estate-project-445516-c4ed0011894f.json'
BQ_TABLE      = 'real-estate-project-445516.real_estate_data.ads_data'
LAKEHOUSE_DIR = '/lakehouse/default/Files/training_data'
# ───────────────────────────────────────────────────────────

# client = bigquery.Client.from_service_account_json(GCP_KEY_PATH)

query = f"""
    SELECT *
    FROM `{BQ_TABLE}`
    WHERE `Ngày gia hạn` >= '2026-03-01T00:00:00'
      AND `Ngày gia hạn` <= '2026-03-31T00:00:00'
"""

# Option 1: Load từ BigQuery (nếu có kết nối)
# df = client.query(query).to_dataframe()

# Option 2: Load từ file CSV (offline)
df = pd.read_csv(f'/kaggle/input/datasets/nguyentri2707/ads-data-2026-03/ads_data_2026_03.csv', low_memory=False)

print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head(2)

## 4. Preprocessing, Deduplication & Advanced Outlier Removal

In [ ]:
# ── Filter ──────────────────────────────────────────────────────────
TOP_4_TYPES = ['căn hộ chung cư', 'nhà riêng', 'đất', 'nhà mặt phố']
# TOP_4_TYPES = ['căn hộ chung cư']

df = df[
    (df['Loại quảng cáo'] == 'Bán') &
    (df['Loại BĐS'].isin(TOP_4_TYPES)) &
    (df['Tỉnh thành phố'] == 'Hà Nội')
].copy()

for col in ['Khoảng giá', 'Diện tích', 'Tọa độ x', 'Tọa độ y']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df[df['Khoảng giá'] > 1e8].dropna(subset=['Khoảng giá', 'Diện tích'])
print(f'After initial filter: {len(df):,} rows')

# ── Deduplication theo 15 cột cấu trúc ─────────────────────────────────
GROUP_COLS = [
    'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1', 'Căn góc',
    'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án',
    'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào',
]

df_dedup = (
    df.groupby(GROUP_COLS, dropna=False)
    .agg(
        Price=('Khoảng giá', lambda s: np.mean(np.unique(s[s.notna()])) if s.notna().any() else np.nan),
        Pháp_lý=('Pháp lý', 'first'),
        Nội_thất=('Nội thất', 'first'),
        Địa_chỉ_2=('Địa chỉ 2', 'first'),
        Tiêu_đề=('Tiêu đề', 'first'),
        Mô_tả=('Mô tả', 'first'),
        Phường=('Phường Xã Thị trấn', 'first'),
        Tọa_độ_x=('Tọa độ x', 'first'),
        Tọa_độ_y=('Tọa độ y', 'first'),
    )
    .reset_index()
    .rename(columns={
        'Pháp_lý': 'Pháp lý', 'Nội_thất': 'Nội thất', 'Địa_chỉ_2': 'Địa chỉ 2',
        'Tiêu_đề': 'Tiêu đề', 'Mô_tả': 'Mô tả', 'Phường': 'Phường Xã Thị trấn',
        'Tọa_độ_x': 'Tọa độ x', 'Tọa_độ_y': 'Tọa độ y',
    })
    .dropna(subset=['Price'])
)
print(f'After dedup: {len(df_dedup):,} rows')

# ── Isolation Forest Outlier Removal (thay vì drop top 1% đơn giản) ──────────
print('Applying Isolation Forest for intelligent outlier detection...')

df_if = df_dedup.copy()
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.02,  # ~2% outliers
    random_state=42,
    n_jobs=-1
)

outlier_mask_all = pd.Series(True, index=df_dedup.index)
for (loai, quan), g in df_dedup.groupby(['Loại BĐS', 'Quận']):
    if len(g) < 10:
        continue
    # Use log-price as features for Isolation Forest
    features_if = np.log1p(g[['Price']].values)
    preds = iso_forest.fit_predict(features_if)
    outlier_mask_all.loc[g.index] = (preds == 1)

df_final = df_dedup[outlier_mask_all].reset_index(drop=True)
print(f'After Isolation Forest: {len(df_final):,} rows')

display(df_final[['Price', 'Diện tích', 'Loại BĐS', 'Quận']].describe())

## 5. Feature Engineering (Nâng cao)

In [ ]:
METRO_STATIONS = [
    (21.028, 105.828), (21.015, 105.820), (21.015, 105.810),
    (21.030, 105.800), (21.002, 105.815), (20.975, 105.776),
]
CENTER_LAT, CENTER_LON = 21.0285, 105.8542

def extract_features(df):
    """
    Extract all features: structural, NLP (V11+V12+V13), spatial, and engineered.
    """
    print('Normalizing text and extracting features...')
    df = df.copy()
    df['clean_desc'] = df['Mô tả'].astype(str).apply(text_normalize).str.lower()
    desc = df['clean_desc']

    if 'Tiêu đề' in df.columns:
        df['clean_title'] = df['Tiêu đề'].astype(str).apply(text_normalize).str.lower()
        text = df['clean_title'] + ' ' + desc
    else:
        df['clean_title'] = ''
        text = desc

    # ── NLP features - V11 legacy ───────────────────────────────────────
    df['feat_oto']        = text.str.contains('xe hơi|ô tô|oto').astype(int)
    df['feat_tranh']      = text.str.contains('tránh').astype(int)
    df['feat_no_hau']     = text.str.contains('nở hậu').astype(int)
    df['feat_thang_may']  = text.str.contains('thang máy').astype(int)
    df['feat_kinh_doanh'] = text.str.contains('kinh doanh|buôn bán').astype(int)
    df['feat_mat_tien']   = text.str.contains('mặt phố|mặt đường').astype(int)
    df['feat_noi_that']   = text.str.contains('nội thất|đầy đủ|tiện nghi').astype(int)
    df['feat_so_do']      = text.str.contains('sổ đỏ|sổ hồng').astype(int)
    df['feat_chinh_chu']  = text.str.contains('chính chủ').astype(int)

    # ── NLP features - V11 new ──────────────────────────────────────
    df['feat_view_nui']      = text.str.contains('view núi|view đồi').astype(int)
    df['feat_view_ho_song']  = text.str.contains('view hồ|view sông|sát hồ|ven hồ').astype(int)
    df['feat_view_canh_dong']= text.str.contains('view cánh đồng|cánh đồng').astype(int)
    df['feat_khuon_vien']    = text.str.contains('sẵn ao|vườn cây|cây ăn trái|nhà vườn').astype(int)
    df['feat_nghi_duong']    = text.str.contains('nghỉ dưỡng|homestay|farmstay|villa').astype(int)
    df['feat_nha_xuong']     = text.str.contains('nhà xưởng').astype(int)
    df['feat_phan_lo']       = text.str.contains('phân lô').astype(int)
    df['feat_f0']            = text.str.contains('f0|chưa qua đầu tư').astype(int)
    df['feat_san_nha']       = text.str.contains('sẵn nhà|nhà cấp 4|ở ngay').astype(int)
    df['feat_duong_nhua']    = text.str.contains('đường nhựa').astype(int)
    df['feat_duong_betong']  = text.str.contains('đường bê tông').astype(int)
    df['feat_truc_chinh']    = text.str.contains('trục chính|đường tỉnh|tỉnh lộ|quốc lộ|đường lớn').astype(int)
    df['feat_phap_ly_chuan'] = text.str.contains('sẵn sổ|sang tên luôn|pháp lý chuẩn').astype(int)
    df['feat_du_lich']       = text.str.contains('khu du lịch|resort').astype(int)
    df['feat_truong_hoc']    = text.str.contains('trường học').astype(int)
    df['feat_cho']           = text.str.contains('chợ').astype(int)
    df['feat_nga_ba_tu']     = text.str.contains('ngã 3|ngã 4|ngã tư').astype(int)

    # ── NLP features - V12 ──────────────────────────────────────
    df['feat_view_bien']     = text.str.contains('biển').astype(int)
    df['feat_goc']           = text.str.contains('góc').astype(int)
    df['feat_cong_vien']     = text.str.contains('công viên').astype(int)
    df['feat_sieu_thi']      = text.str.contains('siêu thị|mart').astype(int)
    df['feat_benh_vien']     = text.str.contains('bệnh viện|bv').astype(int)
    df['feat_tttm']          = text.str.contains('trung tâm thương mại|tttm').astype(int)

    # ── House quality ────────────────────────────────────────
    df['feat_nha_moi']       = text.str.contains('mới xây|xây mới|nhà mới|mới tinh|mới hoàn thiện|bàn giao mới').astype(int)
    df['feat_cai_tao']       = text.str.contains('cải tạo|sửa chữa|nâng cấp|sơn sửa|tu sửa').astype(int)
    df['feat_nha_cu']        = text.str.contains('nhà cũ|cũ kỹ|xuống cấp|cần sửa|cũ nát').astype(int)

    # ── Multiple frontages ─────────────────────────────────
    df['feat_nhieu_mat_tien'] = text.str.contains('2 mặt tiền|3 mặt tiền|hai mặt tiền|mặt tiền trước sau|2 mặt phố|hai mặt phố|thông 2 ngả|thông 2 đường|thông ra 2 ngõ|thông 2 mặt phố').astype(int)
    df['feat_mat_tien_sau']   = text.str.contains('mặt tiền sau|mặt sau|mặt tiền phụ|ngõ thông').astype(int)

    # ── V13: Bổ sung NLP features ─────────────────────────
    df['feat_chung_cu_cao_cap'] = text.str.contains('chung cư cao cấp|chung cư mini|penthouse|duplex|sky villa').astype(int)
    df['feat_ho_boi']           = text.str.contains('hồ bơi|bể bơi').astype(int)
    df['feat_gym']              = text.str.contains('gym|fitness|tập gym|phòng tập').astype(int)
    df['feat_ban_cong']         = text.str.contains('ban công|loggia|sân thượng').astype(int)
    df['feat_an_ninh']          = text.str.contains('an ninh|bảo vệ|bảo vệ 24/24|an ninh tốt|camera').astype(int)
    df['feat_de_xe']            = text.str.contains('chỗ để xe|hầm để xe|tầng hầm').astype(int)
    df['feat_moi_gioi']         = text.str.contains('môi giới|call|liên hệ|hotline').astype(int)
    df['feat_gia_tot']          = text.str.contains('giá tốt|giá rẻ|giá mềm|giá đầu tư|rẻ nhất').astype(int)
    df['feat_thuong_luong']     = text.str.contains('thương lượng|có thương lượng|còn thương lượng').astype(int)

    # ── V14: Bổ sung NLP features dựa trên phân tích định lượng ─────────
    # NOTE: feat_thang_may, feat_no_hau, feat_phan_lo đã có trong V11
    # NOTE: feat_view_ho_song (V11) đã bao gồm view hồ, sát hồ, ven hồ
    # NOTE: feat_goc (V12) đã bao gồm góc -> chỉ thêm features HOÀN TOÀN mới
    df['feat_pho_co']           = (text.str.contains('phố cổ') | text.str.contains('hoàn kiếm')).astype(int)
    df['feat_oto_tranh']        = text.str.contains('ô tô tránh|oto tránh').astype(int)
    df['feat_kinh_doanh_tot']   = text.str.contains('kinh doanh tốt|kinh doanh sầm uất|kinh doanh đỉnh').astype(int)
    df['feat_san_vuon']         = text.str.contains('sân vườn|khuôn viên').astype(int)
    df['feat_quy_hoach_on_dinh']= text.str.contains('quy hoạch ổn định|quy hoạch phân khu').astype(int)
    df['feat_gia_re_cat_lo']    = text.str.contains('giá rẻ|cắt lỗ|bán gấp|cần bán gấp').astype(int)
    df['feat_so_do_cat_ket']    = text.str.contains('cất két|sổ đỏ cất két|sổ cất két').astype(int)
    df['feat_dan_tri_an_sinh']  = text.str.contains('dân trí cao|an sinh đỉnh|an ninh tốt|an sinh tốt').astype(int)

    # ── Geographic distance ───────────────────────────────
    df['dist_to_metro'] = df.apply(
        lambda r: min(np.sqrt((r['Tọa độ x'] - mx)**2 + (r['Tọa độ y'] - my)**2)
                      for mx, my in METRO_STATIONS), axis=1
    )
    df['dist_to_center'] = np.sqrt(
        (df['Tọa độ x'] - CENTER_LAT)**2 + (df['Tọa độ y'] - CENTER_LON)**2
    )

    # ── Combined key ────────────────────────────────────
    df['type_dist'] = df['Loại BĐS'].astype(str) + '_' + df['Quận'].astype(str)

    # ── Clean numeric columns ──────────────────────────────
    all_cols = ['Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Mặt tiền', 'Đường vào']
    int_cols = ['Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh']

    for col in all_cols:
        converted = pd.to_numeric(
            df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce'
        )
        if col in int_cols:
            df[col] = converted.astype(float)  # float thay vi Int64 de tranh pd.NA (CatBoost khong support)
        else:
            df[col] = converted

    df['can_goc'] = df['Căn góc'].astype(str).str.lower().isin(
        ['có', 'yes', '1', 'true', 'căn góc']
    ).astype(int)

    # ── Engineered features ──────────────────────────────
    df['dien_tich_per_tang'] = df['Diện tích'] / df['Số tầng'].replace(0, np.nan).fillna(1)
    df['mat_tien_x_tang']    = df['Mặt tiền'] * df['Số tầng']
    df['log_dien_tich']      = np.log1p(df['Diện tích'])
    df['vuong_area']         = np.sqrt(df['Diện tích'])
    # price_per_m2 KHONG duoc dua vao features vi bi data leakage (duoc tinh truc tiep tu Price - target)
    # Chi dung price_per_m2 de tinh ward-level statistics sau khi split
    df['price_per_m2']       = df['Price'] / df['Diện tích']

    return df

df_final = extract_features(df_final)


## 5b. TF-IDF + SVD: Deep Text Features từ Mô tả

In [ ]:
print('Building TF-IDF + SVD text features...')

full_text = (df_final['clean_title'] + ' ' + df_final['clean_desc']).fillna('')

tfidf = TfidfVectorizer(
    max_features=2000,
    stop_words=None,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)
tfidf_matrix = tfidf.fit_transform(full_text)
print(f'  TF-IDF matrix shape: {tfidf_matrix.shape}')

N_SVD_COMPONENTS = 50
svd = TruncatedSVD(n_components=N_SVD_COMPONENTS, random_state=42)
svd_features = svd.fit_transform(tfidf_matrix)
print(f'  SVD explained variance ratio: {svd.explained_variance_ratio_.sum():.3f}')

for i in range(N_SVD_COMPONENTS):
    df_final[f'desc_svd_{i}'] = svd_features[:, i]



In [ ]:
df_final.info()

## 6. Spatial Clustering & Train/Test Split

In [ ]:
NLP_FEATURES_V11 = [
    'feat_oto', 'feat_tranh', 'feat_no_hau', 'feat_thang_may', 'feat_kinh_doanh',
    'feat_mat_tien', 'feat_noi_that', 'feat_so_do', 'feat_chinh_chu',
    'feat_view_nui', 'feat_view_ho_song', 'feat_view_canh_dong', 'feat_khuon_vien',
    'feat_nghi_duong', 'feat_nha_xuong', 'feat_phan_lo', 'feat_f0', 'feat_san_nha',
    'feat_duong_nhua', 'feat_duong_betong', 'feat_truc_chinh', 'feat_phap_ly_chuan',
    'feat_du_lich', 'feat_truong_hoc', 'feat_cho', 'feat_nga_ba_tu',
    'feat_view_bien', 'feat_goc', 'feat_cong_vien', 'feat_sieu_thi',
    'feat_benh_vien', 'feat_tttm',
    'feat_nha_moi', 'feat_cai_tao', 'feat_nha_cu',
    'feat_nhieu_mat_tien', 'feat_mat_tien_sau',
]

NLP_FEATURES_V13 = [
    'feat_chung_cu_cao_cap', 'feat_ho_boi', 'feat_gym', 'feat_ban_cong',
    'feat_an_ninh', 'feat_de_xe', 'feat_moi_gioi', 'feat_gia_tot', 'feat_thuong_luong',
]

NLP_FEATURES_V14 = [
    'feat_pho_co', 'feat_oto_tranh', 'feat_kinh_doanh_tot',
    'feat_san_vuon', 'feat_quy_hoach_on_dinh', 'feat_gia_re_cat_lo',
    'feat_so_do_cat_ket', 'feat_dan_tri_an_sinh',
]

SVD_FEATURES = [f'desc_svd_{i}' for i in range(N_SVD_COMPONENTS)]

STRUCTURAL_FEATURES = [
    'Loại BĐS', 'Quận', 'Địa chỉ 1', 'Diện tích',
    'Tọa độ x', 'Tọa độ y', 'Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh',
    'Mặt tiền', 'Đường vào', 'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất',
    'can_goc',
]

ENGINEERED_FEATURES = [
    'dist_to_metro', 'dist_to_center', 'type_dist', 'loc_cluster',
    'dien_tich_per_tang', 'mat_tien_x_tang',
    'log_dien_tich', 'vuong_area',
]

CATEGORICAL = [
    'Loại BĐS', 'Quận', 'Địa chỉ 1',
    'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất',
    'type_dist', 'loc_cluster'
]

print('Creating spatial clusters...')
coords = df_final[['Tọa độ x', 'Tọa độ y']].copy()
coords['Tọa độ x'] = coords['Tọa độ x'].fillna(coords['Tọa độ x'].median())
coords['Tọa độ y'] = coords['Tọa độ y'].fillna(coords['Tọa độ y'].median())

kmeans = MiniBatchKMeans(n_clusters=400, random_state=42, n_init=3)
df_final['loc_cluster'] = kmeans.fit_predict(coords)

ALL_FEATURES = (
    STRUCTURAL_FEATURES +
    NLP_FEATURES_V11 +
    NLP_FEATURES_V13 +
    NLP_FEATURES_V14 +
    SVD_FEATURES +
    ENGINEERED_FEATURES
)

print(f'Total features: {len(ALL_FEATURES)}')
print(f'  - Structural: {len(STRUCTURAL_FEATURES)}')
print(f'  - NLP V11: {len(NLP_FEATURES_V11)}')
print(f'  - NLP V13: {len(NLP_FEATURES_V13)}')
print(f'  - NLP V14: {len(NLP_FEATURES_V14)}')
print(f'  - SVD text: {len(SVD_FEATURES)}')


In [ ]:
# ── Prepare X, y ────────────────────────────────────────────
X = df_final[ALL_FEATURES + ['clean_title', 'clean_desc']].copy()
y = np.log1p(df_final['Price'])
groups = df_final['loc_cluster'].values

print(f'Full dataset: {X.shape}')

# ── Train/Test split ──────────────────────────────────────
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.15, random_state=42,
)

# ── Ward-level statistics (chỉ từ TRAIN để tránh leakage) ──────────
# Tính median PPM (Price Per M2) theo từng phường
train_ward_vals = X_train['Địa chỉ 1'].copy()
train_ward_ppm = df_final.loc[X_train.index, 'price_per_m2']
ward_ppm_median = train_ward_ppm.groupby(train_ward_vals.values).transform('median')
X_train['ward_ppm_med'] = ward_ppm_median.fillna(train_ward_ppm.median())

# Áp dụng cho test: map từ phường → median từ train
ward_ppm_map = df_final.loc[X_train.index].groupby('Địa chỉ 1')['price_per_m2'].median().to_dict()
X_test['ward_ppm_med'] = X_test['Địa chỉ 1'].map(ward_ppm_map).fillna(train_ward_ppm.median())

ALL_FEATURES = ALL_FEATURES + ['ward_ppm_med']

print(f'Train: {X_train.shape}, Test: {X_test.shape}')


In [ ]:
encoder = ce.TargetEncoder(cols=CATEGORICAL)
X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc  = encoder.transform(X_test)

print(f'Train encoded: {X_train_enc.shape}')
print(f'Test encoded: {X_test_enc.shape}')

## 7. Optuna Tuning — XGBoost (Nâng cao) - With GPU Support

In [ ]:
N_TRIALS = 30

def objective_xgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 2000, 5000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.03, log=True),
        'max_depth':         trial.suggest_int('max_depth', 8, 16),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 15),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 10.0),
        'tree_method': GPU_CONFIG['xgb_tree_method'],
        'device': GPU_CONFIG['xgb_device'],
        'verbosity': 0,
    }
    
    kf = GroupKFold(n_splits=3)
    scores = []
    
    X_train_num = X_train_enc.drop(columns=['clean_title', 'clean_desc'])
    for _, (train_idx, val_idx) in enumerate(kf.split(X_train_num, y_train, groups_train)):
        X_tr, X_val = X_train_num.iloc[train_idx], X_train_num.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        m = xgb.XGBRegressor(**params, early_stopping_rounds=50, eval_metric='mape')
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        
        scores.append(mean_absolute_percentage_error(
            np.expm1(y_val), np.expm1(m.predict(X_val))
        ))
    
    return float(np.mean(scores))

print(f'Starting XGBoost tuning with {N_TRIALS} trials (3-fold CV each)...')
study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ XGB best CV MAPE: {study_xgb.best_value*100:.2f}%')

## 8. Optuna Tuning — LightGBM (Nâng cao) - With GPU Support

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 2000, 5000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.03, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 255, 1023),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 10.0),
        'min_split_gain':   trial.suggest_float('min_split_gain', 0.0, 1.0),
        'max_bin':          trial.suggest_int('max_bin', 255, 255) if GPU_CONFIG['lgb_device'] == 'gpu' else trial.suggest_int('max_bin', 255, 512),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'device': GPU_CONFIG['lgb_device'],
        'verbose': -1,
    }
    
    kf = GroupKFold(n_splits=3)
    scores = []
    
    X_train_num = X_train_enc.drop(columns=['clean_title', 'clean_desc'])
    for train_idx, val_idx in kf.split(X_train_num, y_train, groups_train):
        X_tr, X_val = X_train_num.iloc[train_idx], X_train_num.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='mape',
              callbacks=[lgb.log_evaluation(0)])
        
        scores.append(mean_absolute_percentage_error(
            np.expm1(y_val), np.expm1(m.predict(X_val))
        ))
    
    return float(np.mean(scores))

print(f'Starting LightGBM tuning with {N_TRIALS} trials (3-fold CV each)...')
study_lgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ LGB best CV MAPE: {study_lgb.best_value*100:.2f}%')


## 8b. Optuna Tuning — CatBoost - With GPU Support

In [ ]:
print('Starting CatBoost tuning...')

def objective_cat(trial):
    params = {
        'iterations':       trial.suggest_int('iterations', 1000, 3000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth':            trial.suggest_int('depth', 6, 12),
        'l2_leaf_reg':      trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'border_count':     trial.suggest_int('border_count', 128, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength':  trial.suggest_float('random_strength', 0.0, 1.0),
        'verbose': 0,
        'loss_function': 'RMSE',
        'eval_metric': 'MAPE',
        'od_type': 'Iter',
        'od_wait': 50,
        'task_type': GPU_CONFIG['cat_task_type'],
    }
    
    kf = GroupKFold(n_splits=3)
    scores = []
    
    # Prepare text columns
    X_train_cat = X_train_enc.copy()
    X_train_cat['clean_title'] = X_train_cat['clean_title'].fillna("").astype(str)
    X_train_cat['clean_desc'] = X_train_cat['clean_desc'].fillna("").astype(str)
    
    for train_idx, val_idx in kf.split(X_train_cat, y_train, groups_train):
        X_tr, X_val = X_train_cat.iloc[train_idx], X_train_cat.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        m = CatBoostRegressor(**params, text_features=['clean_title', 'clean_desc'])
        m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
        
        scores.append(mean_absolute_percentage_error(
            np.expm1(y_val), np.expm1(m.predict(X_val))
        ))
    
    return float(np.mean(scores))

study_cat = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_cat.optimize(objective_cat, n_trials=min(N_TRIALS, 15), show_progress_bar=True)

print(f'\n✅ CatBoost best CV MAPE: {study_cat.best_value*100:.2f}%')


## 9. Final Training & Advanced Ensemble (Stacking) - With GPU Support

In [ ]:
print('=' * 60)
print('  V14: Training Final Models + Stacking Ensemble')
print(f'  Device: GPU_CONFIG (XGB: {GPU_CONFIG["xgb_tree_method"]}, LGB: {GPU_CONFIG["lgb_device"]}, Cat: {GPU_CONFIG["cat_task_type"]})')
print('=' * 60)

# Drop text columns for XGBoost/LightGBM
X_train_xgb = X_train_enc.drop(columns=['clean_title', 'clean_desc'])
X_test_xgb = X_test_enc.drop(columns=['clean_title', 'clean_desc'])

# ── 1. Train XGBoost ──────────────────────────────────────
print('\n[1/3] Training final XGBoost...')
xgb_params = study_xgb.best_params.copy()
xgb_params['tree_method'] = GPU_CONFIG['xgb_tree_method']
xgb_params['device'] = GPU_CONFIG['xgb_device']
m_xgb = xgb.XGBRegressor(**xgb_params, verbosity=0)
m_xgb.fit(X_train_xgb, y_train)
p_xgb = np.expm1(m_xgb.predict(X_test_xgb))

# ── 2. Train LightGBM ──────────────────────────────────
print('[2/3] Training final LightGBM...')
lgb_params = study_lgb.best_params.copy()
lgb_params['device'] = GPU_CONFIG['lgb_device']
m_lgb = lgb.LGBMRegressor(**lgb_params, verbose=-1)
m_lgb.fit(X_train_xgb, y_train)
p_lgb = np.expm1(m_lgb.predict(X_test_xgb))

# ── 3. Train CatBoost ─────────────────────────────────
print('[3/3] Training final CatBoost...')
X_train_cat = X_train_enc.copy()
X_train_cat['clean_title'] = X_train_cat['clean_title'].fillna("").astype(str)
X_train_cat['clean_desc'] = X_train_cat['clean_desc'].fillna("").astype(str)

X_test_cat = X_test_enc.copy()
X_test_cat['clean_title'] = X_test_cat['clean_title'].fillna("").astype(str)
X_test_cat['clean_desc'] = X_test_cat['clean_desc'].fillna("").astype(str)

cat_params = study_cat.best_params.copy()
cat_params['task_type'] = GPU_CONFIG['cat_task_type']
m_cat = CatBoostRegressor(**cat_params, verbose=0, text_features=['clean_title', 'clean_desc'])
m_cat.fit(X_train_cat, y_train, verbose=0)
p_cat = np.expm1(m_cat.predict(X_test_cat))

y_true = np.expm1(y_test)

# ── Method 1: Grid search weighted blend (3 models) ─────────────
print('\nOptimizing ensemble weights (grid search)...')
best_mape_blend = 999
best_w_xgb, best_w_lgb, best_w_cat = 0.34, 0.33, 0.33
for w1 in np.arange(0.0, 1.01, 0.1):
    for w2 in np.arange(0.0, 1.01 - w1, 0.1):
        w3 = 1.0 - w1 - w2
        p_blend = w1 * p_xgb + w2 * p_lgb + w3 * p_cat
        mape = mean_absolute_percentage_error(y_true, p_blend)
        if mape < best_mape_blend:
            best_mape_blend = mape
            best_w_xgb, best_w_lgb, best_w_cat = w1, w2, w3

print(f'  Best blend weights: XGB={best_w_xgb:.1f}, LGB={best_w_lgb:.1f}, Cat={best_w_cat:.1f}')
print(f'  Blend MAPE: {best_mape_blend*100:.2f}%')

# ── Method 2: Stacking with Ridge meta-model ─────────────────────
print('\nTraining stacking ensemble with Ridge meta-model...')
X_stack_train = np.column_stack([
    np.expm1(m_xgb.predict(X_train_xgb)),
    np.expm1(m_lgb.predict(X_train_xgb)),
    np.expm1(m_cat.predict(X_train_cat)),
])
X_stack_test = np.column_stack([p_xgb, p_lgb, p_cat])

meta_model = Ridge(alpha=1.0, random_state=42)
meta_model.fit(X_stack_train, y_true)
p_stack = meta_model.predict(X_stack_test)
mape_stack = mean_absolute_percentage_error(y_true, p_stack)
print(f'  Stacking MAPE: {mape_stack*100:.2f}%')

# ── Final prediction: use the better of blend vs stacking ───────
if mape_stack < best_mape_blend:
    print('\n✅ Stacking outperforms simple blend, using stacking as final model')
    p_final = p_stack
    final_mape = mape_stack
else:
    print('\n✅ Simple blend outperforms stacking, using blend as final model')
    p_final = best_w_xgb * p_xgb + best_w_lgb * p_lgb + best_w_cat * p_cat
    final_mape = best_mape_blend

print(f'\n📊 Final Test MAPE: {final_mape*100:.2f}%')
print(f'📊 Final Test R²:  {r2_score(y_true, p_final):.4f}')

## 10. SHAP Feature Selection & Interpretation

In [ ]:
import shap

print('Computing SHAP values for feature importance analysis...')

# Use XGBoost for SHAP analysis (fastest with tree explainer)
explainer = shap.TreeExplainer(m_xgb)
X_train_sample = X_train_xgb.sample(min(1000, len(X_train_xgb)), random_state=42)
shap_values = explainer.shap_values(X_train_sample)

# ── SHAP Feature Importance ──────────────────────────────────────
shap_importance = np.abs(shap_values).mean(axis=0)
feature_names = X_train_xgb.columns
shap_df = pd.DataFrame({'feature': feature_names, 'importance': shap_importance})
shap_df = shap_df.sort_values('importance', ascending=False)

print(f'\nTop 20 most important features (by SHAP):')
for i, row in shap_df.head(20).iterrows():
    print(f'  {i+1:3d}. {row["feature"]:40s} {row["importance"]:.6f}')

# ── Auto feature selection: remove features with near-zero importance ──
importance_threshold = shap_importance.max() * 0.001  # 0.1% of max importance
important_features = feature_names[shap_importance > importance_threshold]
removed_features = feature_names[shap_importance <= importance_threshold]
print(f'\nAuto feature selection:')
print(f'  Features kept: {len(important_features)}')
print(f'  Features removed (near-zero SHAP): {len(removed_features)}')
if len(removed_features) > 0:
    print(f'  Removed: {list(removed_features)}')

# ── Summary plot ────────────────────────────────────────────────
shap.summary_plot(shap_values, X_train_sample, max_display=20, show=False)
print('\n✅ SHAP analysis complete')


## 11. Detailed Evaluation & Error Analysis

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

print('=' * 60)
print('  FINAL MODEL EVALUATION')
print('=' * 60)

mae = mean_absolute_error(y_true, p_final)
mse = mean_squared_error(y_true, p_final)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(y_true, p_final)
r2 = r2_score(y_true, p_final)

print(f'  MAE:  {mae:,.0f} VND')
print(f'  RMSE: {rmse:,.0f} VND')
print(f'  MAPE: {mape*100:.2f}%')
print(f'  R²:   {r2:.4f}')
print('=' * 60)

# ── Error by property type ───────────────────────────────────────
print('\n📊 Error by Property Type:')
results_df = X_test_enc[['Loại BĐS']].copy()
results_df['y_true'] = y_true
results_df['y_pred'] = p_final
results_df['error_pct'] = np.abs(results_df['y_true'] - results_df['y_pred']) / results_df['y_true'] * 100

for loai in results_df['Loại BĐS'].unique():
    subset = results_df[results_df['Loại BĐS'] == loai]
    print(f'  {loai:25s}: MAPE={subset["error_pct"].mean():.2f}%, count={len(subset)}')

# ── Error by district ────────────────────────────────────────────
print('\n📊 Error by District (top 10):')
results_df['Quận'] = X_test_enc['Quận']
district_errors = results_df.groupby('Quận')['error_pct'].agg(['mean', 'count']).sort_values('count', ascending=False)
for quan, row in district_errors.head(10).iterrows():
    print(f'  {quan:25s}: MAPE={row["mean"]:.2f}%, count={int(row["count"])}')

print('\n✅ Evaluation complete')


## 12. Save Models & Artifacts

In [ ]:
# import os
# from datetime import datetime

# MODEL_DIR = f'{LAKEHOUSE_DIR}/models_v14_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
# os.makedirs(MODEL_DIR, exist_ok=True)

# print(f'Saving models to {MODEL_DIR}...')

# # ── Save individual models ───────────────────────────────────────
# joblib.dump(m_xgb, f'{MODEL_DIR}/xgb_model.pkl')
# joblib.dump(m_lgb, f'{MODEL_DIR}/lgb_model.pkl')
# m_cat.save_model(f'{MODEL_DIR}/cat_model.cbm')

# # ── Save meta-model and weights ──────────────────────────────────
# joblib.dump(meta_model, f'{MODEL_DIR}/meta_model.pkl')
# np.save(f'{MODEL_DIR}/blend_weights.npy', np.array([best_w_xgb, best_w_lgb, best_w_cat]))

# # ── Save encoders and transformers ───────────────────────────────
# joblib.dump(encoder, f'{MODEL_DIR}/target_encoder.pkl')
# joblib.dump(tfidf, f'{MODEL_DIR}/tfidf_vectorizer.pkl')
# joblib.dump(svd, f'{MODEL_DIR}/svd_transformer.pkl')
# joblib.dump(kmeans, f'{MODEL_DIR}/kmeans_cluster.pkl')

# # ── Save feature lists ───────────────────────────────────────────
# feature_config = {
#     'ALL_FEATURES': ALL_FEATURES,
#     'CATEGORICAL': CATEGORICAL,
#     'STRUCTURAL_FEATURES': STRUCTURAL_FEATURES,
#     'NLP_FEATURES_V11': NLP_FEATURES_V11,
#     'NLP_FEATURES_V13': NLP_FEATURES_V13,
#     'NLP_FEATURES_V14': NLP_FEATURES_V14,
#     'SVD_FEATURES': SVD_FEATURES,
#     'ENGINEERED_FEATURES': ENGINEERED_FEATURES,
#     'N_SVD_COMPONENTS': N_SVD_COMPONENTS,
#     'METRO_STATIONS': METRO_STATIONS,
#     'CENTER_LAT': CENTER_LAT,
#     'CENTER_LON': CENTER_LON,
# }
# with open(f'{MODEL_DIR}/feature_config.json', 'w', encoding='utf-8') as f:
#     json.dump(feature_config, f, ensure_ascii=False, indent=2)

# # ── Save evaluation metrics ──────────────────────────────────────
# metrics = {
#     'mae': float(mae),
#     'rmse': float(rmse),
#     'mape': float(mape),
#     'r2': float(r2),
#     'final_mape_pct': float(final_mape * 100),
#     'blend_weights': [float(best_w_xgb), float(best_w_lgb), float(best_w_cat)],
#     'stacking_mape': float(mape_stack * 100),
#     'blend_mape': float(best_mape_blend * 100),
#     'xgb_best_cv_mape': float(study_xgb.best_value * 100),
#     'lgb_best_cv_mape': float(study_lgb.best_value * 100),
#     'cat_best_cv_mape': float(study_cat.best_value * 100),
#     'n_train': len(X_train),
#     'n_test': len(X_test),
#     'n_features': len(ALL_FEATURES),
#     'gpu_config': GPU_CONFIG,
# }
# with open(f'{MODEL_DIR}/metrics.json', 'w', encoding='utf-8') as f:
#     json.dump(metrics, f, ensure_ascii=False, indent=2)

# print(f'\n✅ All models and artifacts saved to {MODEL_DIR}')
# print(f'✅ Final MAPE: {final_mape*100:.2f}%')
